# 10 · Expert behaviour, without light

**What this shows.** How trained animals behave on the Uniform distribution, from sessions with no light at
all: one animal through the library pipeline, then every animal — the psychometric parameters, the history
statistics, the psychometric curves and the update matrices. These are the same sessions model identification
fits (notebook 11), so this is the behaviour those models have to explain.

**Which sessions.** The `model_id_uniform` preset in `config.yaml`: regular Uniform sessions (no laser, no
masking, no ALM), the last five with at least 70 % correct and 250 responded trials. Animals without such
sessions are listed and left out. Hard-phase behaviour without light exists only for the pre-opto cohort, in
blocks: notebook 12.

**What produced it.** Only the snapshot. This notebook computes point estimates itself (no resampling; a few
seconds), unlike the others, which read report tables. Genotype and cohort are labels here, not tests.

In [ ]:
from nb_setup import *
from behav_utils import (TrialArrays, compute_psychometric_curve, compute_stats, compute_update_matrix, filter_trials,
                         plot_psychometric_curve, plot_update_matrix)
from behav_utils.stats import PSYCHOMETRIC

banner()
exp = load_experiment()
STATS = ['accuracy', 'hard_accuracy', 'easy_accuracy', 'side_bias', *PSYCHOMETRIC, 'recency', 'win_stay', 'lose_shift']
sessions = {aid: expert_sessions(a) for aid, a in sorted(exp.animals.items())}
used = {aid: s for aid, s in sessions.items() if s}
cohort = cohorts_by_animal(sessions)
print(f'{len(used)} of {len(sessions)} animals have expert sessions without light; left out:',
      sorted(set(sessions) - set(used)) or 'none')

## 1 · One animal: the pipeline

`select_sessions → filter_trials → TrialArrays → compute_x → plot_x`, once by hand. The curve's band is a
trial bootstrap of this animal's sessions; everything after this section is point estimates.

In [ ]:
aid = sorted(used)[0]
arrays = TrialArrays.from_sessions(filter_trials(used[aid]))
curve = compute_psychometric_curve(arrays, n_bootstrap=100)
um = compute_update_matrix(arrays)

fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
plot_psychometric_curve(curve, ax=axes[0], color=PALETTE[0])
plot_update_matrix(um, ax=axes[1])
axes[0].set_title(f'{aid}: {len(used[aid])} sessions, {arrays.n_trials} trials')
plt.tight_layout()
compute_stats(arrays, STATS).round(3)

## 2 · Every animal

One row per animal: the statistics of its expert sessions pooled (`docs/stats_reference.md` defines each).
`mu` is the criterion (positive: more evidence needed to choose B), `sigma` the slope, the lapses the
asymptotes; `recency`, `win_stay` and `lose_shift` are the history effects.

In [ ]:
rows, curves, mats = [], {}, {}
for aid, s in used.items():
    arr = TrialArrays.from_sessions(filter_trials(s))
    rows.append({'animal': aid, 'cohort': cohort.get(aid, '?'),
                 'genotype': exp.animals[aid].metadata.get('genotype', 'unknown'),
                 'sessions': len(s), 'trials': arr.n_trials, **compute_stats(arr, STATS).to_dict()})
    curves[aid] = compute_psychometric_curve(arr, n_bootstrap=0)
    mats[aid] = compute_update_matrix(arr)
expert = pd.DataFrame(rows).set_index('animal')
display(expert.round(3))
expert.groupby('cohort')[STATS].median().round(3)

In [ ]:
cohorts = sorted(expert['cohort'].unique())
colour = {c: PALETTE[i % len(PALETTE)] for i, c in enumerate(cohorts)}
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
for aid, c in curves.items():
    plot_psychometric_curve(c, ax=axes[0], color=colour[expert.loc[aid, 'cohort']], alpha=0.5, show_band=False,
                            show_data=False, show_reference=False)
axes[0].set(title='psychometric curves, one per animal', xlabel='stimulus', ylabel='P(choose B)')
rng = np.random.default_rng(0)
params = ['mu', 'sigma', 'lapse_low', 'lapse_high']
for j, p in enumerate(params):
    for c in cohorts:
        v = expert.loc[expert['cohort'] == c, p].to_numpy()
        axes[1].scatter(j + rng.uniform(-0.15, 0.15, len(v)), v, color=colour[c], s=18,
                        label=c if j == 0 else None)
axes[1].axhline(0, color='0.8', lw=0.6)
axes[1].set(xticks=range(len(params)), xticklabels=params, title='fit parameters per animal')
axes[1].legend(frameon=False, fontsize=8)
plt.tight_layout()

## 3 · Update matrices

The update matrix is the shift in P(B | current stimulus bin) by the previous stimulus bin, after a correct
trial. It is the readout the BE and SC models are fitted to in notebook 11. Mean over animals per cohort.

In [ ]:
fig, axes = plt.subplots(1, len(cohorts), figsize=(3.6 * len(cohorts), 3.4), squeeze=False)
for ax, c in zip(axes[0], cohorts):
    m = [mats[aid].matrix for aid in expert.index[expert['cohort'] == c]]
    im = ax.imshow(np.nanmean(m, axis=0), vmin=-0.3, vmax=0.3, cmap='RdBu_r', origin='lower')
    ax.set(title=f'{c} (n={len(m)})', xlabel='previous stimulus bin', ylabel='current stimulus bin')
fig.colorbar(im, ax=axes[0].tolist(), shrink=0.8, label='shift in P(B)')

## Where next

- `11` — which strategy, BE or SC, explains these animals' updates better.
- `12` — what happens when the distribution changes (pre-opto cohort, blocks).
- `20` / `21` — PPC inactivation in experts and on the Hard phases.